In [9]:
from mushroom_rl.environments import LQR
from mushroom_rl.solvers.lqr import *
import jax
jax.config.update('jax_default_matmul_precision', 'float32')

STATE_DIM,A_DIM = 4,2
env = LQR.generate(s_dim=STATE_DIM,a_dim=A_DIM,gamma=0.99,episodic=True,horizon=500,random_init=True)

In [18]:
import numpy as np
size = 10000
env.reset()

dataset = []
for i in range(size):  
    action = np.random.sample(A_DIM)
    obs, reward, done, info = env.step(action)
    dataset.append(obs)
    if done or i%500==0:
        env.reset()
        
dataset = np.array(dataset)
        
    

False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False


0

In [2]:
# %%

import os
import wandb
import argparse
import itertools
import numpy as np
import jax
import jax.numpy as jnp
from jaxrl_m.common import CodeTimer
import logging
import envpool
logging.basicConfig(level=logging.CRITICAL)


def get_batch(i,batches):
    return  jax.tree.map(lambda x: x[i], batches)

def body(i,val):
    agent,batches = val
    return (agent.update_critics(get_batch(i,batches)),batches)

def str2bool(v):
    if isinstance(v, bool):
        return v
    if v.lower() in ('yes', 'true', 't', 'y', '1'):
        return True
    elif v.lower() in ('no', 'false', 'f', 'n', '0'):
        return False
    else:
        raise argparse.ArgumentTypeError('Boolean value expected.')
    

def none_or_str(value):
    if value == 'None':
        return None
    return value

# Set env variables
os.environ["WANDB_API_KEY"]="28996bd59f1ba2c5a8c3f2cc23d8673c327ae230"
os.environ['PYTHONHASHSEED'] = '1'
os.environ['TF_CUDNN_DETERMINISTIC'] = '1'

##############################
parser = argparse.ArgumentParser()

parser.add_argument('--seed',type=int,default=42) 

parser.add_argument('--algo_name', type=str, default='superppo', help='the name of the RL algorithm')
parser.add_argument('--project_name',type=str,default="single_exp") 

parser.add_argument('--env_name',type=str,default="Hopper-v5") 
parser.add_argument('--max_steps',type=int,default=10_000) 
parser.add_argument('--max_episode_steps',type=int,default=500) 
parser.add_argument('--num_rollouts',type=int,default=4) 
parser.add_argument('--gamma',type=float,default=0.99)
parser.add_argument('--healthy_reward',type=float,default=1.) 
parser.add_argument('--entropy_coeff',type=float,default=1.) 

parser.add_argument('--discount_actor',type=str2bool,default=True)
parser.add_argument('--min_target',type=str2bool,default=True)
parser.add_argument('--discount_entropy',type=str2bool,default=True) 
parser.add_argument('--on_policy_data',type=str2bool,default=False)
parser.add_argument('--adaptive_critics',type=str2bool,default=False) 
parser.add_argument('--num_critics',type=int,default=2)

parser.add_argument('--critic_lr',type=float,default=3e-4) 
parser.add_argument('--actor_lr',type=float,default=3e-4) 
parser.add_argument('--temp_lr',type=float,default=3e-4)
parser.add_argument('--use_layer_norm',type=str2bool,default=True)

parser.add_argument('--momentum',type=float,default=0.) 
parser.add_argument('--num_actor_updates',type=int,default=5) 
parser.add_argument('--clipping_ratio',type=float,default=0.1) 
parser.add_argument('--hidden_dims',type=int,default=256) 
parser.add_argument('--episode_based',type=str2bool,default=False) 
parser.add_argument('--tanh_squash_actions',type=str2bool,default=True) 


args = parser.parse_args(args=[])

from jaxrl_m.onsac_clean import *

hidden_dims = ()
NUM_UPDATES = 1000





import os
from functools import partial
import numpy as np
import jax
import tqdm
import gymnasium as gym


from jaxrl_m.wandb import setup_wandb, default_wandb_config, get_flag_dict
import wandb
from jaxrl_m.evaluation import supply_rng, evaluate, flatten, EpisodeMonitor
from jaxrl_m.dataset import ReplayBuffer,ActorReplayBuffer
from collections import deque
from jax import config
from jaxrl_m.utils import flatten_rollouts
from jaxrl_m.evaluate_critic import evaluate_many_critics
from jaxrl_m.rollout import rollout_policy_lqr
from jax import config
config.update("jax_debug_nans", True)

eval_episodes=10
batch_size = 256
max_steps = args.max_steps
start_steps = 0
log_interval = 10000
n_grads = 0

# wandb_config = {
#     'project': args.project_name,
#     'name':None,
#     'hyperparam_dict':args.__dict__,
#     }
#wandb_run = setup_wandb(**wandb_config)


###############""


observation = jnp.ones(env._mdp_info.observation_space.shape)
action = jnp.ones(env._mdp_info.action_space.shape)


example_transition = dict(
    observations=observation,
    actions=action,
    rewards=0.0,
    masks=1.0,
    next_observations=observation,
    pre_actions = action,
    discounts=1.0,
    log_probs=0.,
)
buffer_size = args.num_rollouts*args.max_episode_steps if args.on_policy_data else 100_000
replay_buffer = ReplayBuffer.create(example_transition, size=int(buffer_size))
actor_buffer = ActorReplayBuffer.create(example_transition, size=int(args.num_rollouts*args.max_episode_steps))

agent = create_learner(args.seed,
                    
                observations=example_transition['observations'][None],
                actions =example_transition['actions'][None],
                max_steps=max_steps,
                discount=args.gamma,
                discount_actor=args.discount_actor,
                min_target=args.min_target,
                discount_entropy=args.discount_entropy,
                adaptive_critics=args.adaptive_critics,
                num_critics= args.num_critics,
                entropy_coeff=args.entropy_coeff,
                temp_lr=args.temp_lr,
                actor_lr=args.actor_lr,
                critic_lr=args.critic_lr,
                momentum=args.momentum,
                clipping_ratio=args.clipping_ratio,
                num_actor_updates=args.num_actor_updates,
                critic_hidden_dims=(args.hidden_dims,args.hidden_dims),
                actor_hidden_dims=(),
                use_layer_norm= args.use_layer_norm,
                state_dependent_std=False,
                tanh_squash_distribution=False,
                tanh_squash_actions=False,
                use_bias=False,
                
                )

##############




exploration_metrics = dict()
#obs,info = env.reset()    
exploration_rng = jax.random.PRNGKey(0)
i = 0
unlogged_steps,cached_steps = 0,0
policy_rollouts = deque([], maxlen=20)
warmup = True
R2,bias = jnp.ones(args.num_critics),jnp.zeros(args.num_critics)


with tqdm.tqdm(total=max_steps) as pbar:
    
    while (i < max_steps):
        with jax.log_compiles(False):
            warmup=(i < start_steps)
            
            logging.debug('policy rollout')
            replay_buffer,actor_buffer,policy_rollout,policy_return,variance,undisc_policy_return,num_steps = rollout_policy_lqr(
                                                                    agent,env,exploration_rng,
                                                                    replay_buffer,actor_buffer,eval=False,
                                                                    num_rollouts=args.num_rollouts,discount = args.gamma,max_length=args.max_episode_steps)
            
            print(f'policy_return: {policy_return}, undisc_policy_return {undisc_policy_return}')                                                              
            if not warmup : policy_rollouts.append(policy_rollout)
            unlogged_steps += num_steps
            cached_steps += num_steps
            i+=num_steps
            pbar.update(int(num_steps))
            
            if replay_buffer.size > start_steps:
            
                ### Update critics ###:
                logging.debug('update critics')
                transitions = replay_buffer.get_all()
                idxs = jax.random.choice(agent.rng,a=transitions['observations'].shape[0], shape=(NUM_UPDATES,256), replace=True)
                batches = jax.vmap(lambda i: jax.tree.map(lambda x: x[i], transitions))(idxs)
                agent = agent.update_critics_seq(batches,R2)
                
                
                
                
                ### Update actor ###
                actor_batch = actor_buffer.get_all()    
                agent, actor_update_info = agent.update_actor(actor_batch,R2)    
                critic_update_info = {}
                update_info = {**critic_update_info, **actor_update_info}
                n_grads += 1
                
                ### Grad stuff ###
                def flatten(grads):    
                    tmp = jax.tree.map(lambda x: jnp.reshape(x,(-1,)),grads)
                    tmp = jax.tree_util.tree_flatten(tmp)[0]
                    tmp = jnp.concatenate(tmp)
                    return tmp

                #one = flatten(grads)
                
                # print(f'gradient approx {one}')
                # print('cosine distance',jnp.dot(one.flatten(),two.flatten())/(jnp.linalg.norm(one.flatten())*jnp.linalg.norm(two.flatten())))
                # wandb.log({'cosine_distance':jnp.dot(one.flatten(),two.flatten())/(jnp.linalg.norm(one.flatten())*jnp.linalg.norm(two.flatten()))}, step=int(i),commit=False)

            
                    
                
                ### Log training info ###
                exploration_metrics = {f'exploration/disc_return': policy_return,'training/std': jnp.sqrt(variance)}
                train_metrics = {f'training/{k}': v for k, v in update_info.items()}
                train_metrics['training/undisc_return'] = undisc_policy_return
                                    
            
                if cached_steps >= int(1e6): 
                    jax.clear_caches()
                    cached_steps = 0
                    print('clearing cache')
        


/home/mahdi/Desktop/supersac/.venv/lib/python3.10/site-packages/wandb/util.py:152: SentryHubDeprecationWarning: `sentry_sdk.Hub` is deprecated and will be removed in a future major release. Please consult our 1.x to 2.x migration guide for details on how to migrate `Hub` usage to the new API: https://docs.sentry.io/platforms/python/migration/1.x-to-2.x
  sentry_hub = sentry_sdk.Hub(sentry_client)
2024-09-15 20:37:11.725567: W external/xla/xla/service/gpu/nvptx_compiler.cc:836] The NVIDIA driver's CUDA version is 12.5 which is older than the PTX compiler version (12.6.68). Because the driver is older than the PTX compiler version, XLA is disabling parallel compilation, which may slow down compilation. You should update your NVIDIA driver or use the NVIDIA-provided CUDA forward compatibility packages.


Extra kwargs: {'max_steps': 10000}


 20%|██        | 2000/10000 [00:01<00:04, 1847.43it/s]

policy_return: -5950.7175233428825, undisc_policy_return -41762.03766200055


 40%|████      | 4000/10000 [00:06<00:11, 543.57it/s] 

...

 80%|████████  | 8000/10000 [00:12<00:03, 659.22it/s]

policy_return: -7291.34969149525, undisc_policy_return -45474.98196508112


100%|██████████| 10000/10000 [00:13<00:00, 824.39it/s]

policy_return: -9512.69385608617, undisc_policy_return -58823.56856005


100%|██████████| 10000/10000 [00:14<00:00, 702.23it/s]


In [5]:
import copy

rng = jax.random.PRNGKey(42)


new_agent = create_learner(args.seed,
                        
                    observations=example_transition['observations'][None],
                    actions =example_transition['actions'][None],
                    max_steps=max_steps,
                    discount=args.gamma,
                    discount_actor=args.discount_actor,
                    min_target=args.min_target,
                    discount_entropy=args.discount_entropy,
                    adaptive_critics=args.adaptive_critics,
                    num_critics= args.num_critics,
                    entropy_coeff=args.entropy_coeff,
                    temp_lr=args.temp_lr,
                    actor_lr=args.actor_lr,
                    critic_lr=args.critic_lr,
                    momentum=args.momentum,
                    clipping_ratio=args.clipping_ratio,
                    num_actor_updates=args.num_actor_updates,
                    critic_hidden_dims=(args.hidden_dims,args.hidden_dims),
                    actor_hidden_dims=(),
                    use_layer_norm= args.use_layer_norm,
                    state_dependent_std=False,
                    tanh_squash_distribution=False,
                    tanh_squash_actions=False,
                    use_bias=False,
                    
                    )


new_agent.actor.params['log_stds'] = -100 * jnp.ones_like(new_agent.actor.params['log_stds'])
new_agent.actor.params['means']['kernel'] = jnp.copy(agent.actor.params['means']['kernel'])
print(agent.actor.params['log_stds'])
print(new_agent.actor.params['log_stds'])
new_agent = new_agent.update_critics_seq(batches,R2)




Extra kwargs: {'max_steps': 10000}
[0.00709894 0.00653   ]
[-100. -100.]


In [4]:
jax.config.update("jax_enable_x64", True)
jax.config.update('jax_default_matmul_precision', 'highest')
obs = env.reset()

K = np.array(agent.actor.params['means']['kernel'])
#noise = jnp.exp(agent.actor.params['log_stds']['kernel'])# For state dependant noise but i think formulation is without
agent.actor.params['log_stds']= -100 * jnp.ones_like(tmp)
noise = jnp.diag(jnp.exp(agent.actor.params['log_stds'])) # For state independant noise
K_ref= compute_lqr_feedback_gain(env)
#agent.actor.params['means']['kernel'] = K.
########################################
reference = compute_lqr_V(obs,env,K_ref)

V = compute_lqr_V(obs,env,K.T)
V_noise = compute_lqr_V_gaussian_policy(obs,env,K.T,noise)
reference_noise = compute_lqr_V_gaussian_policy(obs,env,K_ref,noise)
#Q = compute_lqr_V_gaussian_policy(obs,action,env,K_ref,1*np.ones_like(K_ref).T)
print(f'Reference {reference} Reference_noise {reference_noise}')

total = 0
gamma = 1
exploration_rng = jax.random.PRNGKey(52)

print(f'V {V} V_noise {V_noise}')

for i in range(500):

    exploration_rng, key = jax.random.split(exploration_rng)
    action,_,_ = agent.sample_actions(obs,seed=exploration_rng)
    #action = K.T@obs
    next_obs, reward, done, info = env.step(action)  
    obs = next_obs
    total+=gamma*reward
    gamma *= 0.99
    
print(total)


NameError: name 'tmp' is not defined

In [ ]:
original = agent.actor.params['log_stds']

In [ ]:
K = compute_lqr_feedback_gain(env)
Sigma = np.ones_like(K).T

# K = np.array(agent.actor.params['means']['kernel']).T
# Sigma = np.exp(agent.actor.params["log_stds"]['kernel'])

for i in range(500):
    obs = env.reset()
    compute_lqr_Q_gaussian_policy_gradient_K(obs,K@obs,env,K,Sigma)
#compute_lqr_V(obs,env,K)